In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import seaborn as sns

In [54]:
# Set the default template globally
pio.templates.default = "plotly_white"

#Reads the dataframe from memory
df = pd.read_csv('../data/processed/rail_accident_pop.xls', low_memory = False)
#Display all columns
pd.set_option('display.max_columns', None)

In [55]:
df['Date'] = pd.to_datetime(df.loc[:, 'Date'])
df['Time'] = pd.to_datetime(df.loc[:, 'Time'])
df['Hour'] = df['Time'].dt.hour
df['Persons Evacuated'] = df['Persons Evacuated'].str.replace(',', '').astype(int)
df['Gross Tonnage'] = df['Gross Tonnage'].str.replace(',', '').astype(int)
df['Equipment Damage Cost'] = df.loc[:, 'Equipment Damage Cost'].str.replace(',', '').astype(float)
df['Track Damage Cost'] = df.loc[:, 'Track Damage Cost'].str.replace(',', '').astype(float)
df['Total Damage Cost'] = df.loc[:, 'Total Damage Cost'].str.replace(',', '').astype(float)

C:\Users\Timothy Mai\AppData\Local\Temp\ipykernel_11284\2896567986.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df.loc[:, 'Time'])


In [56]:
features = [
            'Reporting Railroad Code', 'Accident Month', 'Hour', 
            'Accident Type Code', 'Hazmat Cars', 'Hazmat Cars Damaged', 'Hazmat Released Cars', 'Persons Evacuated', 'State Code', 
            'Temperature', 'Visibility Code', 'Weather Condition Code', 'Track Type Code', 'Track Class', 'Train Direction Code', 
            'Equipment Type Code', 'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'Signalization Code', 'Causing Car Position',
            'Head End Locomotives', 'Mid Train Manual Locomotives', 'Mid Train Remote Locomotives', 'Rear End Manual Locomotives', 
            'Rear End Remote Locomotives', 'Derailed Head End Locomotives', 'Derailed Mid Train Manual Locomotives',
            'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives', 'Derailed Rear End Remote Locomotives',
            'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives', 'Derailed Rear End Remote Locomotives',
            'Loaded Freight Cars', 'Loaded Passenger Cars', 'Empty Freight Cars', 'Empty Passenger Cars', 'Cabooses',
            'Derailed Loaded Freight Cars', 'Derailed Loaded Passenger Cars', 'Derailed Empty Freight Cars', 
            'Derailed Empty Passenger Cars', 'Derailed Cabooses', 'Accident Cause Code', 
            'Engineers On Duty', 'Firemen On Duty', 'Conductors On Duty', 'Brakemen On Duty',
            'Railroad Employees Killed', 'Railroad Employees Injured', 'Passengers Killed', 'Passengers Injured', 
            'Others Killed', 'Others Injured', 'Total Persons Killed', 'Total Persons Injured',
            'Joint Track Class', 'Class Code', 'Class', 'FIPS', 'Population_2020', 'RUCC_2023', 'Location_Class', 'Cause_Category'
           ]

#targets = ['Equipment Damage Cost', 'Track Damage Cost', 'Total Damage Cost']
targets = ['Total Damage Cost']
#Consider adding: Equipment Attended (bool), Passengers Transported (bool), Hours/Minutes Conductors/Engineers On Duty

corr_df = df[features + targets]
corr_df = corr_df.dropna()

#Factorizes all non-numeric columns
corr_df.loc[:, list(corr_df.dtypes == 'object')] = corr_df.loc[:, list(corr_df.dtypes == 'object')].apply(lambda x: pd.factorize(x)[0])


In [57]:
factor_keys = df.loc[:, list(df.dtypes == 'object')].apply(lambda x: pd.factorize(x)[1])
cause_categories = factor_keys['Cause_Category']

In [76]:
pd.set_option('display.max_rows', None)

corr_features = {}

for i in range(5):
    print(cause_categories[i])
    corr_features[cause_categories[i]] = corr_df[corr_df['Cause_Category'] == i].corr()['Total Damage Cost'].nlargest(6)
    display(corr_features[cause_categories[i]][1:])
    print()

pd.reset_option('display.max_rows')

T


Derailed Loaded Freight Cars    0.544316
Passengers Killed               0.473038
Passengers Injured              0.452416
Maximum Speed                   0.435206
Train Speed                     0.425841
Name: Total Damage Cost, dtype: float64


M


Derailed Loaded Freight Cars     0.285816
Persons Evacuated                0.208663
Hazmat Cars Damaged              0.125866
Derailed Head End Locomotives    0.120037
Derailed Empty Freight Cars      0.112559
Name: Total Damage Cost, dtype: float64


H


Total Persons Killed              0.686923
Total Persons Injured             0.619981
Passengers Injured                0.548166
Passengers Killed                 0.525057
Derailed Loaded Passenger Cars    0.522602
Name: Total Damage Cost, dtype: float64


E


Derailed Loaded Freight Cars    0.665806
Hazmat Released Cars            0.384632
Hazmat Cars Damaged             0.329969
Loaded Freight Cars             0.276738
Gross Tonnage                   0.262204
Name: Total Damage Cost, dtype: float64


S


Total Persons Injured             0.593153
Passengers Injured                0.526532
Derailed Loaded Passenger Cars    0.492655
Maximum Speed                     0.478226
Train Speed                       0.393806
Name: Total Damage Cost, dtype: float64

| Code | Category | Examples |
|------|----------|----------|
| H | Human Factors | Operator error, procedure violations, switching mistakes |
| M | Mechanical & Electrical | Equipment failures, brake defects, mechanical breakdowns |
| T | Track | Track geometry defects, rail defects, track maintenance issues |
| S | Signal | Signal system failures, communication errors |
| E | Miscellaneous/Environmental | Weather, vandalism, other external factors |

| Cause Category | Code / Category | 
|---|----|
| 0 | Track (T) |
| 1 | Mechanical (M) |
| 2 | Human Factors (H) |
| 3 | Miscellaneous/Environmental (E) |
| 4 | Signal (S) |

In [59]:
# Checking avg value of each feature with Total Damage Cost

corr_df.groupby("Cause_Category")[[
    "Train Speed",
    "Derailed Loaded Freight Cars",
    "Passengers Injured",
    "Hazmat Released Cars",
    "Total Damage Cost"
]].mean()

,Train Speed,Derailed Loaded Freight Cars,Passengers Injured,Hazmat Released Cars,Total Damage Cost
Cause_Category,,,,,
0,10.467398,3.996242,0.019540,0.038930,215919.012626
1,24.299889,1.045290,0.164855,0.014075,171634.663880
2,5.797790,0.977413,0.065080,0.007150,127342.159409
3,17.683638,2.008690,0.002997,0.021876,215187.259215
4,5.671916,0.709974,0.013123,0.003937,83655.653543


## Comparing Weather Conditions

In [80]:
# Checking avg of weather condition codes vs number of derailed

corr_df.groupby("Weather Condition Code")["Derailed Loaded Freight Cars"].mean().sort_values(ascending=False)

Weather Condition Code
5.0    2.076923
3.0    2.024379
6.0    1.827526
2.0    1.777645
1.0    1.733128
4.0    1.608844
Name: Derailed Loaded Freight Cars, dtype: float64

| Weather Condition Code | Value | 
|---|----|
| 1 | clear |
| 2 | cloudy |
| 3 | rain |
| 4 | fog |
| 5 | sleet |
| 6 | snow |

In [81]:
# Comparing Weather Condition Code to Total Cost

corr_df.groupby("Weather Condition Code")["Total Damage Cost"].mean().sort_values(ascending=False)

Weather Condition Code
3.0    227635.781060
1.0    163060.682830
2.0    157801.005828
6.0    148266.322300
5.0    144820.400000
4.0    143149.724490
Name: Total Damage Cost, dtype: float64

## Comparing Visibility / Speed

In [82]:
corr_df.groupby("Visibility Code")["Derailed Loaded Freight Cars"].mean().sort_values(ascending=False)

Visibility Code
1.0    1.857597
3.0    1.815442
4.0    1.759541
2.0    1.736215
Name: Derailed Loaded Freight Cars, dtype: float64

| Visibility Code | Value | 
|---|----|
| 1 | dawn |
| 2 | day |
| 3 | dusk |
| 4 | dark |

In [83]:
corr_df.groupby("Visibility Code")["Train Speed"].mean().sort_values(ascending=False)

Visibility Code
2.0    14.027891
1.0    11.824221
4.0    11.310703
3.0    10.848967
Name: Train Speed, dtype: float64

In [84]:
corr_df.groupby("Visibility Code")["Derailed Loaded Freight Cars"].mean()

Visibility Code
1.0    1.857597
2.0    1.736215
3.0    1.815442
4.0    1.759541
Name: Derailed Loaded Freight Cars, dtype: float64

In [73]:
corr_df.groupby("Visibility Code")["Total Damage Cost"].mean().sort_values(ascending=False)

Visibility Code
1.0    194670.262528
4.0    169715.272266
2.0    162480.254592
3.0    151931.493333
Name: Total Damage Cost, dtype: float64

## Crew Staffing 

In [85]:
corr_df["No_Brakemen"] = (corr_df["Brakemen On Duty"] == 0).astype(int)

corr_df.groupby("No_Brakemen")[[
    "Train Speed",
    "Derailed Loaded Freight Cars",
    "Total Persons Injured",
    "Total Damage Cost"
]].mean()


,Train Speed,Derailed Loaded Freight Cars,Total Persons Injured,Total Damage Cost
No_Brakemen,,,,
0,13.568682,1.313876,0.274868,109067.642519
1,12.200152,1.905239,0.166580,183642.327652
